# IDAC Data Preprocessing

This notebook performs data preprocessing operations on the IDAC Excel dataset. You should run this prior to working with data in Tableau if you need any data preprocessing.

Behavior is controlled via environment variables specified in a .env file placed outside this directory. 
A sample .env file is included - you'll need to modify it prior to running this notebook.

In [1]:
from dotenv import load_dotenv

loaded_vars = load_dotenv(verbose=True)
if not loaded_vars:
    raise Exception('Unable to load any environment variables - make sure you have a .env file')


## Load Dataset

Load Excel dataset into memory. All sheets will be read, so this operation may take awhile.

Requires `IDAC_DATASET_PATH` environment variable to be set, like in .env file.

Ex: `IDAC_DATASET_PATH = folder1/folder2/filename.xlsx` will load the Excel file at "folder1/folder2/filename.xlsx".

In [2]:
import pandas as pd
import os

idac_dataset = pd.read_excel(os.getenv('IDAC_DATASET_PATH'), sheet_name=None)

idac_dataset.keys()

dict_keys(['Appeals', 'Juvenile Cases Filed & Dispo', 'Adult Cases Filed', 'Adult Cases Dispo'])

## Operation 0: Split and Combine Sheets

**About:**
This operation splits the Juvenile Cases Filed & Dispo sheet into separate datasets for cases filed and cases disposed, then combines them with their respective adult sheets, e.g. Adult Cases Filed for cases filed dataset. It produces 2 new sheets: Cases Filed (for adult + juvenile cases filed), and Cases Disposed (for adult + juvenile cases disposed).

**Notes:**
- This operation is not optional
- It will always be executed first, and subsequent operations will depend on its results
- For Cases Disposed, because LEV Simple Code and LEV Simple Code_recode store different values depending on adult/juvenile, a new field Disposition Type is created which combines the values of both

In [3]:
import pandas as pd
from IPython.display import display

juvenile_df = idac_dataset['Juvenile Cases Filed & Dispo']

juvenile_filed = juvenile_df.loc[juvenile_df['Metric Type'] == 'Cases Filed']
juvenile_dispo = juvenile_df.loc[juvenile_df['Metric Type'] == 'Cases Disposed']

juvenile_dispo['Disposition Type'] = juvenile_dispo['LEV Simple Code'].copy()

adult_filed = idac_dataset['Adult Cases Filed']
adult_dispo = idac_dataset['Adult Cases Dispo']

adult_dispo['Disposition Type'] = adult_dispo['LEV Simple Code_recode'].copy()

all_filed = pd.concat([adult_filed,juvenile_filed], axis=0)
all_dispo = pd.concat([adult_dispo,juvenile_dispo], axis=0)

idac_dataset['Cases Filed'] = all_filed
idac_dataset['Cases Dispo'] = all_dispo

display(all_filed.head(n=2))
display(all_dispo.head(n=2))

del idac_dataset['Juvenile Cases Filed & Dispo']
del idac_dataset['Adult Cases Filed']
del idac_dataset['Adult Cases Dispo']

display(idac_dataset.keys())


,Metric Type,Court Path Origination,Unit of Analysis,Court System,County,Year,LEV Simple Code,Representation Timeframe,Count of cases,Public Defender,...,OTN with more than 1 AttorneyRepresentationType,CSPCaseType,Case Type Recode,OffenseGrade,Status,CaseCategory,OffenseGrade_recode,OffenseGrade_reode_With_H,Data Date,County_Class
0,Cases Filed,Criminal,Unique OTN,MDJS,Adams,2017,NaN,Any Point in Case Lifecycle,179,60,...,0.0,DUI,DUI,M,Closed,NaN,M,M,7/22/2025,Fifth Class
1,Cases Filed,Criminal,Unique OTN,MDJS,Adams,2017,NaN,Any Point in Case Lifecycle,68,26,...,2.0,Property,Property,M,Closed,NaN,M,M,7/22/2025,Fifth Class


,Metric Type,Court Path Origination,Unit of Analysis,Court System,County,Year,LEV Simple Code,LEV Simple Code_recode,Representation Timeframe,Count of cases,...,CSPCaseType,Case Type Recode,OffenseGrade,Status,CaseCategory,Data Date,County_Class,Secondary Disposition Code,Disposition Type,Post Disposition Attorney
0,Cases Disposed,Criminal,Unique OTN,CPCMS,Adams,2017,Closed - Unknown/NULL Dispostion,Adjudicated or Closed with Unknown Disposition,Only At Disposition,26,...,Drugs,Drugs,M,Closed,NaN,7/22/2025,Fifth Class,NaN,Adjudicated or Closed with Unknown Disposition,NaN
1,Cases Disposed,Criminal,Unique OTN,CPCMS,Adams,2017,Closed - Unknown/NULL Dispostion,Adjudicated or Closed with Unknown Disposition,Only At Disposition,1,...,DUI,DUI,M,Closed,NaN,7/22/2025,Fifth Class,NaN,Adjudicated or Closed with Unknown Disposition,NaN


dict_keys(['Appeals', 'Cases Filed', 'Cases Dispo'])

## Operation 1: Remove Empty Non-Value Columns

**About:**
This operation removes blank/empty non-value columns in specified sheets. Only non-value columns that are entirely empty are removed. We define value columns to be those that store the counts, e.g. Public Defender or Court Appointed.


**Environment Variables:**
Requires `DO_OP1_APPEALS`, `DO_OP1_FILED`, `DO_OP1_DISPO` environment variables to be set to either "y" or "n". If "y", the operation will be performed on the sheet. If "n", the operation will not be performed on the sheet.

Ex: `DO_OP1_FILED= y` will perform empty column removal on Cases Filed sheet.

**Notes:**
- Any missing/null values in the value columns will be filled with 0.

In [4]:
import pandas as pd

ALL_VAL_COLS = [
    'Count of cases',
    'Public Defender',
    'Court Appointed',
    'Private',
    'Other',
    'No indication of defendant representation type',
    'No Attorney Present',
    'Waived',
    'Unknown',
    'N/A: No Formal Hearing Held',
    'Involving indigent defender',
    'Pro Se',
]

def zerofill_if_col_empty(df:pd.DataFrame, cols:list[str]):
    for col in cols:
        if df[col].isnull().all():
            df[col] = df[col].fillna(value=0).astype(int)

    return df

def find_empty_columns(df:pd.DataFrame) -> list[str]:
    empty = []

    for col in df.columns:
        if df[col].isnull().all():
            empty.append(col)

    return empty

In [5]:
import pandas as pd
import os

if os.getenv('DO_OP1_APPEALS').lower() == 'y':
    print('Performing empty non-value column removal on Appeals sheet')

    appeals_df = idac_dataset['Appeals']

    appeals_df = zerofill_if_col_empty(appeals_df, ALL_VAL_COLS)

    drop_cols = find_empty_columns(appeals_df)
    print(f'The following columns will be removed: {drop_cols}')

    appeals_df = appeals_df.drop(columns=drop_cols)
    idac_dataset['Appeals'] = appeals_df

    print(f'Remaining columns in Appeals sheet: {appeals_df.columns}')
    

Performing empty non-value column removal on Appeals sheet
The following columns will be removed: ['LEV Simple Code', 'Homicide', 'CSPCaseType', 'Case Type Recode', 'OffenseGrade', 'Status']
Remaining columns in Appeals sheet: Index(['Metric Type', 'Court Path Origination', 'Unit of Analysis',
       'Court System', 'County', 'Year', 'Representation Timeframe',
       'Count of cases', 'Public Defender', 'Court Appointed', 'Private',
       'Other', 'No indication of defendant representation type',
       'No Attorney Present', 'Waived', 'Unknown',
       'N/A: No Formal Hearing Held', 'Involving indigent defender', 'Pro Se',
       'OTN with more than 1 AttorneyRepresentationType', 'CaseCategory',
       'Data Date', 'County_Class'],
      dtype='str')


In [6]:
import pandas as pd
import os

if os.getenv('DO_OP1_FILED').lower() == 'y':
    print('Performing empty column removal on Cases Filed sheet')

    filed_df = idac_dataset['Cases Filed']

    filed_df = zerofill_if_col_empty(filed_df, ALL_VAL_COLS)

    drop_cols = find_empty_columns(filed_df)
    print(f'The following columns will be removed: {drop_cols}')

    filed_df = filed_df.drop(columns=drop_cols)
    idac_dataset['Cases Filed'] = filed_df

    print(f'Remaining columns in Cases Filed sheet: {filed_df.columns}')


Performing empty column removal on Cases Filed sheet
The following columns will be removed: ['LEV Simple Code', 'Post Disposition Attorney', 'CaseCategory']
Remaining columns in Cases Filed sheet: Index(['Metric Type', 'Court Path Origination', 'Unit of Analysis',
       'Court System', 'County', 'Year', 'Representation Timeframe',
       'Count of cases', 'Public Defender', 'Court Appointed', 'Private',
       'Other', 'No indication of defendant representation type',
       'No Attorney Present', 'Waived', 'Unknown',
       'N/A: No Formal Hearing Held', 'Involving indigent defender', 'Pro Se',
       'Homicide', 'OTN with more than 1 AttorneyRepresentationType',
       'CSPCaseType', 'Case Type Recode', 'OffenseGrade', 'Status',
       'OffenseGrade_recode', 'OffenseGrade_reode_With_H', 'Data Date',
       'County_Class'],
      dtype='str')


In [7]:
import pandas as pd
import os

if os.getenv('DO_OP1_DISPO').lower() == 'y':
    print('Performing empty column removal on Cases Dispo sheet')

    dispo_df = idac_dataset['Cases Dispo']

    dispo_df = zerofill_if_col_empty(dispo_df, ALL_VAL_COLS)

    drop_cols = find_empty_columns(dispo_df)
    print(f'The following columns will be removed: {drop_cols}')

    dispo_df = dispo_df.drop(columns=drop_cols)
    idac_dataset['Cases Dispo'] = dispo_df

    print(f'Remaining columns in Cases Dispo sheet: {dispo_df.columns}')


Performing empty column removal on Cases Dispo sheet
The following columns will be removed: ['CaseCategory', 'Secondary Disposition Code']
Remaining columns in Cases Dispo sheet: Index(['Metric Type', 'Court Path Origination', 'Unit of Analysis',
       'Court System', 'County', 'Year', 'LEV Simple Code',
       'LEV Simple Code_recode', 'Representation Timeframe', 'Count of cases',
       'Public Defender', 'Court Appointed', 'Private', 'Other',
       'No indication of defendant representation type', 'No Attorney Present',
       'Waived', 'Unknown', 'N/A: No Formal Hearing Held',
       'Involving indigent defender', 'Pro Se', 'Homicide',
       'OTN with more than 1 AttorneyRepresentationType', 'CSPCaseType',
       'Case Type Recode', 'OffenseGrade', 'Status', 'Data Date',
       'County_Class', 'Disposition Type', 'Post Disposition Attorney'],
      dtype='str')


## Operation 2: Convert to Long Format


**About:**
The raw IDAC dataset is wide. Computers work better/more efficiently with long data.

This operation transforms each sheet to be in long format. I**t assumes the value columns are those of the counts for each sheet**, e.g. Public Defender, Court Appointed, Private, etc. **The other assumption is that the (row) identifier columns are ALL the non-value columns**, e.g. County, Year, Data Date, etc. Naturally, this is sheet-specific. Two new columns are created: Count Type, which stores the type of count variable (e.g. Public Defender), and Count, which stores the actual numeric count of the count variable for that row (e.g. 101).

**Environment Variables:**
Requires `DO_OP2_APPEALS`, `DO_OP2_FILED`, and `DO_OP2_DISPO`  environment variables to be set to either "y" or "n". If "y", the operation will be performed on the sheet. If "n", the operation will not be performed on the sheet.

Ex: `DO_OP2_FILED = y` will perform wide-to-long format conversion on Cases Filed sheet.

**Notes:**
- Any "missing"/NaN values in numeric columns will be filled with 0 during this operation (to enforce data consistency)

In [8]:
import pandas as pd
from IPython.display import display

if os.getenv('DO_OP2_FILED').lower() == 'y':
    filed_df = idac_dataset['Cases Filed']
    
    numeric_columns = filed_df.select_dtypes(include=['number']).columns
    filed_df[numeric_columns] = filed_df[numeric_columns].fillna(value=0).astype(int)

    id_cols = [col for col in filed_df.columns.to_list() if col not in ALL_VAL_COLS]

    var_name = 'Count Type'
    val_name = 'Count'

    print(f'Identifier columns: {id_cols}')
    print(f'Value columns: {ALL_VAL_COLS}')

    # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    long_filed_df = pd.melt(
        filed_df,
        id_vars=id_cols,
        value_vars=ALL_VAL_COLS,
        var_name=var_name,
        value_name=val_name
    )

    display(long_filed_df.head(n=5))

    idac_dataset['Cases Filed'] = long_filed_df


Identifier columns: ['Metric Type', 'Court Path Origination', 'Unit of Analysis', 'Court System', 'County', 'Year', 'Representation Timeframe', 'Homicide', 'OTN with more than 1 AttorneyRepresentationType', 'CSPCaseType', 'Case Type Recode', 'OffenseGrade', 'Status', 'OffenseGrade_recode', 'OffenseGrade_reode_With_H', 'Data Date', 'County_Class']
Value columns: ['Count of cases', 'Public Defender', 'Court Appointed', 'Private', 'Other', 'No indication of defendant representation type', 'No Attorney Present', 'Waived', 'Unknown', 'N/A: No Formal Hearing Held', 'Involving indigent defender', 'Pro Se']


,Metric Type,Court Path Origination,Unit of Analysis,Court System,County,Year,Representation Timeframe,Homicide,OTN with more than 1 AttorneyRepresentationType,CSPCaseType,Case Type Recode,OffenseGrade,Status,OffenseGrade_recode,OffenseGrade_reode_With_H,Data Date,County_Class,Count Type,Count
0,Cases Filed,Criminal,Unique OTN,MDJS,Adams,2017,Any Point in Case Lifecycle,0,0,DUI,DUI,M,Closed,M,M,7/22/2025,Fifth Class,Count of cases,179
1,Cases Filed,Criminal,Unique OTN,MDJS,Adams,2017,Any Point in Case Lifecycle,0,2,Property,Property,M,Closed,M,M,7/22/2025,Fifth Class,Count of cases,68
2,Cases Filed,Criminal,Unique OTN,MDJS,Adams,2017,Any Point in Case Lifecycle,0,0,Weapons,Public Order/Other,M,Closed,M,M,7/22/2025,Fifth Class,Count of cases,5
3,Cases Filed,Criminal,Unique OTN,MDJS,Adams,2017,Any Point in Case Lifecycle,0,0,Motor Vehicle-Other,Public Order/Other,M,Closed,M,M,7/22/2025,Fifth Class,Count of cases,8
4,Cases Filed,Criminal,Unique OTN,MDJS,Adams,2017,Any Point in Case Lifecycle,0,1,Property,Property,F,Closed,F,F,7/22/2025,Fifth Class,Count of cases,56


In [9]:
import pandas as pd
from IPython.display import display

if os.getenv('DO_OP2_DISPO').lower() == 'y':
    dispo_df = idac_dataset['Cases Dispo']
    
    numeric_columns = dispo_df.select_dtypes(include=['number']).columns
    dispo_df[numeric_columns] = dispo_df[numeric_columns].fillna(value=0).astype(int)

    id_cols = [col for col in dispo_df.columns.to_list() if col not in ALL_VAL_COLS]

    var_name = 'Count Type'
    val_name = 'Count'

    print(f'Identifier columns: {id_cols}')
    print(f'Value columns: {ALL_VAL_COLS}')

    # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    long_dispo_df = pd.melt(
        dispo_df,
        id_vars=id_cols,
        value_vars=ALL_VAL_COLS,
        var_name=var_name,
        value_name=val_name
    )

    display(long_dispo_df.head(n=5))

    idac_dataset['Cases Dispo'] = long_dispo_df


Identifier columns: ['Metric Type', 'Court Path Origination', 'Unit of Analysis', 'Court System', 'County', 'Year', 'LEV Simple Code', 'LEV Simple Code_recode', 'Representation Timeframe', 'Homicide', 'OTN with more than 1 AttorneyRepresentationType', 'CSPCaseType', 'Case Type Recode', 'OffenseGrade', 'Status', 'Data Date', 'County_Class', 'Disposition Type', 'Post Disposition Attorney']
Value columns: ['Count of cases', 'Public Defender', 'Court Appointed', 'Private', 'Other', 'No indication of defendant representation type', 'No Attorney Present', 'Waived', 'Unknown', 'N/A: No Formal Hearing Held', 'Involving indigent defender', 'Pro Se']


,Metric Type,Court Path Origination,Unit of Analysis,Court System,County,Year,LEV Simple Code,LEV Simple Code_recode,Representation Timeframe,Homicide,...,CSPCaseType,Case Type Recode,OffenseGrade,Status,Data Date,County_Class,Disposition Type,Post Disposition Attorney,Count Type,Count
0,Cases Disposed,Criminal,Unique OTN,CPCMS,Adams,2017,Closed - Unknown/NULL Dispostion,Adjudicated or Closed with Unknown Disposition,Only At Disposition,0,...,Drugs,Drugs,M,Closed,7/22/2025,Fifth Class,Adjudicated or Closed with Unknown Disposition,0,Count of cases,26
1,Cases Disposed,Criminal,Unique OTN,CPCMS,Adams,2017,Closed - Unknown/NULL Dispostion,Adjudicated or Closed with Unknown Disposition,Only At Disposition,0,...,DUI,DUI,M,Closed,7/22/2025,Fifth Class,Adjudicated or Closed with Unknown Disposition,0,Count of cases,1
2,Cases Disposed,Criminal,Unique OTN,CPCMS,Adams,2017,Closed - Unknown/NULL Dispostion,Adjudicated or Closed with Unknown Disposition,Only At Disposition,0,...,Person,Person,M,Closed,7/22/2025,Fifth Class,Adjudicated or Closed with Unknown Disposition,0,Count of cases,2
3,Cases Disposed,Criminal,Unique OTN,CPCMS,Adams,2017,Closed - Unknown/NULL Dispostion,Adjudicated or Closed with Unknown Disposition,Only At Disposition,0,...,Public Order,Public Order/Other,M,Closed,7/22/2025,Fifth Class,Adjudicated or Closed with Unknown Disposition,0,Count of cases,4
4,Cases Disposed,Criminal,Unique OTN,CPCMS,Adams,2018,Closed - Unknown/NULL Dispostion,Adjudicated or Closed with Unknown Disposition,Only At Disposition,0,...,Drugs,Drugs,M,Closed,7/22/2025,Fifth Class,Adjudicated or Closed with Unknown Disposition,0,Count of cases,36


In [10]:
import pandas as pd
from IPython.display import display

if os.getenv('DO_OP2_APPEALS').lower() == 'y':
    appeals_df = idac_dataset['Appeals']

    id_cols = [col for col in appeals_df.columns.to_list() if col not in ALL_VAL_COLS]

    var_name = 'Count Type'
    val_name = 'Count'

    print(f'Identifier columns: {id_cols}')
    print(f'Value columns: {ALL_VAL_COLS}')

    # see https://pandas.pydata.org/docs/reference/api/pandas.melt.html
    long_appeals_df = pd.melt(
        appeals_df,
        id_vars=id_cols,
        value_vars=ALL_VAL_COLS,
        var_name=var_name,
        value_name=val_name
    )

    display(long_appeals_df.head(n=5))

    idac_dataset['Appeals'] = long_appeals_df


Identifier columns: ['Metric Type', 'Court Path Origination', 'Unit of Analysis', 'Court System', 'County', 'Year', 'Representation Timeframe', 'OTN with more than 1 AttorneyRepresentationType', 'CaseCategory', 'Data Date', 'County_Class']
Value columns: ['Count of cases', 'Public Defender', 'Court Appointed', 'Private', 'Other', 'No indication of defendant representation type', 'No Attorney Present', 'Waived', 'Unknown', 'N/A: No Formal Hearing Held', 'Involving indigent defender', 'Pro Se']


,Metric Type,Court Path Origination,Unit of Analysis,Court System,County,Year,Representation Timeframe,OTN with more than 1 AttorneyRepresentationType,CaseCategory,Data Date,County_Class,Count Type,Count
0,Cases Appealed,Criminal,Unique Docket,Supreme,Huntingdon,2021,Any Point in Case Lifecycle,0,Criminal,11/14/2025,Sixth Class,Count of cases,2
1,Cases Appealed,Criminal,Unique Docket,Superior,Huntingdon,2023,Any Point in Case Lifecycle,0,Criminal,11/14/2025,Sixth Class,Count of cases,18
2,Cases Appealed,Criminal,Unique Docket,Superior,Huntingdon,2022,Any Point in Case Lifecycle,0,Criminal,11/14/2025,Sixth Class,Count of cases,17
3,Cases Appealed,Criminal,Unique Docket,Supreme,Huntingdon,2020,Any Point in Case Lifecycle,0,Criminal,11/14/2025,Sixth Class,Count of cases,12
4,Cases Appealed,Criminal,Unique Docket,Superior,Huntingdon,2020,Any Point in Case Lifecycle,0,Criminal,11/14/2025,Sixth Class,Count of cases,26


## Save to Disk

**About:**
With all data preprocessing operations complete, last step is to save data to a file on disk. 

Excel has a max limit of 1,048,576 rows per sheet, which may be exceeded depending on preprocessing and dataset size. In that case, data will be saved to multiple CSV files instead.

**Environment Variables:**
If `MAKE_COUNTIES_CSV` is set to "y", then an additional CSV file will be generated containing the unique counties and the state they are associated with (PA).

If `MAKE_COUNT_TYPES_CSV` is set to "y", then an additional CSV file will be generated containing the unique count types across the datasets.

In [11]:
EXCEL_MAX_ROWS = 1048576
save_xlsx = True # prefer 1 excel workbook w/ multiple sheets by default
for sheet_name, df in idac_dataset.items():
    print(f'Sheet {sheet_name} has {len(df)} rows')

    if len(df) > EXCEL_MAX_ROWS:
        save_xlsx = False

print(f'Saving as Excel: {save_xlsx}')

Sheet Appeals has 14196 rows
Sheet Cases Filed has 889872 rows
Sheet Cases Dispo has 1332372 rows
Saving as Excel: False


In [12]:
import time
import pandas as pd
from datetime import datetime

if save_xlsx:
    print('Saving as 1 Excel workbook w/ multiple sheets')

    fname = f'IDAC Data Long {datetime.now().strftime('%m%d%y%H%M%S')}.xlsx'
    
    file_start = time.time()
    with pd.ExcelWriter(fname, engine='xlsxwriter') as xlsx_writer:
        for sheet_name, df in idac_dataset.items():
            sheet_start = time.time()
            print(f'Began writing sheet {sheet_name} to {fname}')
            df.to_excel(xlsx_writer, sheet_name=sheet_name, engine='xlsxwriter', index=False)
            print(f'Finished writing sheet {sheet_name} to {fname} in {time.time()-sheet_start} seconds')

    print(f'Saved to {fname} in {time.time()-file_start} seconds')

In [13]:
import pandas as pd
import time
from datetime import datetime

if not save_xlsx:
    print('Saving as multiples CSVs - 1 per sheet')

    ftime = datetime.now().strftime('%m%d%y-%H%M%S')
    files_start = time.time()
    for sheet_name, df in idac_dataset.items():
        sheet_start = time.time()
        fname = f'IDAC {sheet_name} Data Long {ftime}.csv'

        print(f'Began writing sheet {sheet_name} to {fname}')
        df.to_csv(fname, index=False)
        print(f'Finished writing sheet {sheet_name} to {fname} in {time.time()-sheet_start} seconds')

    print(f'Saved CSVs to disk in {time.time()-files_start} seconds')


Saving as multiples CSVs - 1 per sheet
Began writing sheet Appeals to IDAC Appeals Data Long 030826-100301.csv
Finished writing sheet Appeals to IDAC Appeals Data Long 030826-100301.csv in 0.09786295890808105 seconds
Began writing sheet Cases Filed to IDAC Cases Filed Data Long 030826-100301.csv
Finished writing sheet Cases Filed to IDAC Cases Filed Data Long 030826-100301.csv in 7.0607452392578125 seconds
Began writing sheet Cases Dispo to IDAC Cases Dispo Data Long 030826-100301.csv
Finished writing sheet Cases Dispo to IDAC Cases Dispo Data Long 030826-100301.csv in 12.702128410339355 seconds
Saved CSVs to disk in 19.86134696006775 seconds


In [14]:
import os
from IPython.display import display

if os.getenv('MAKE_COUNTIES_CSV').lower() == 'y':
    print('Creating counties CSV file...')

    counties = idac_dataset['Cases Filed']['County'].unique()
    states = ['PA' for _ in range(len(counties))]

    df = pd.DataFrame.from_dict({'County':counties, 'State':states})
    display(df.head(n=3))

    fname = f'IDAC PA Counties {ftime}.csv'
    df.to_csv(fname, index=False)


Creating counties CSV file...


,County,State
0,Adams,PA
1,Allegheny,PA
2,Armstrong,PA


In [15]:
import os
from IPython.display import display

if os.getenv('MAKE_COUNT_TYPES_CSV').lower() == 'y':
    print('Creating count types CSV file...')

    df = pd.DataFrame.from_dict({'Count Type':ALL_VAL_COLS})
    display(df.head(n=3))

    fname = f'IDAC Count Types {ftime}.csv'
    df.to_csv(fname, index=False)

Creating count types CSV file...


,Count Type
0,Count of cases
1,Public Defender
2,Court Appointed
